In [ ]:
# ==============================================================================
# CELL 0: HARDWARE, ENVIRONMENT & WHEELHOUSE INSTALLATION
# ==============================================================================
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

_CANDS = [
    Path('/kaggle/input/gemma-4-developer-agent-wheelhouse'),
    Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
]
WHEELHOUSE_DIR = next((p for p in _CANDS if p.exists() and list(p.glob('*.whl'))), None)
if WHEELHOUSE_DIR is None:
    _found = sorted(Path('/kaggle/input').rglob('swegemma*.whl'))
    assert _found, 'no swegemma wheel anywhere under /kaggle/input'
    WHEELHOUSE_DIR = _found[0].parent
print('USING WHEELHOUSE:', WHEELHOUSE_DIR)

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')


In [ ]:
# ==============================================================================
# CELL 1: CONFIGURE TEST AGENT (SETTING 1: THINKING MINIMAL + SETTING 2: PSEUDO-XML)
# ==============================================================================
import os
import shutil
from pathlib import Path
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'test_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)

# 1. Setting 1: generate_content_config disabling thinking and CoT loops to prevent 500 error
sampling_yaml = """temperature: 0.1
max_output_tokens: 16384
thinking_config:
  thinking_level: "minimal"
  include_thoughts: false
  thinking_budget: 0
"""
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(sampling_yaml, encoding='utf-8')

# 2. Setting 2: Pseudo-XML boundary so Gemma treats instruction as strict programmatic directive
prompt_md = """<SYSTEM_DIRECTIVE_CRITICAL>
You are an elite Software Engineering AI operating inside an isolated repository sandbox at `/workspace`.
Your ONLY goal is to solve the provided GitHub issue and submit a verified patch.

RULES YOU MUST FOLLOW:
1. REPRODUCE FIRST: Run a python command or pytest via `run_command` to observe the failure.
2. INSPECT SURGICALLY: Read relevant files using `read_file` with precise offset and limit.
3. EDIT WITH PRECISION: Use `edit_file` to replace exact text. Never duplicate function signatures or classes.
4. VERIFY LOCALLY: Run pytest via `run_command` to ensure reproduction passes and zero regressions occur.
5. SUBMIT: Call `submit_patch` as your final tool call.
</SYSTEM_DIRECTIVE_CRITICAL>

<USER_ISSUE_BELOW>
Pay attention to the user query that immediately follows this block:
"""
(AGENT_DIR / 'prompts' / 'main.md').write_text(prompt_md, encoding='utf-8')

# 3. agent.yaml with native 6 clean tools (run_command, read_file, edit_file, write_file, get_status, submit_patch)
agent_yaml = """name: main
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/main.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
generate_content_config: !include configs/sampling.yaml
"""
(AGENT_DIR / 'agent.yaml').write_text(agent_yaml, encoding='utf-8')

# 4. eval_config.yaml
eval_yaml = """eval_budget:
  max_duration_seconds: 240
  max_tool_calls: 30
  max_turns: 50
"""
(AGENT_DIR / 'eval_config.yaml').write_text(eval_yaml, encoding='utf-8')

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

print(f'Test agent configured at {AGENT_DIR}')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  - {p.relative_to(AGENT_DIR)}: {p.stat().st_size} bytes')


In [ ]:
# ==============================================================================
# CELL 2: START OFFLINE VLLM SERVER (TENSOR PARALLEL ON 4x L4 GPUs)
# ==============================================================================
import litellm
import torch
from pathlib import Path
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
_MP = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not _MP.exists():
    _f = sorted(Path('/kaggle/input').rglob('gemma-4-31b-it-qat-w4a16-ct'))
    print('model fallback search:', _f[:3])
    _MP = _f[0] if _f else _MP
MODEL_PATH = _MP
print('USING MODEL:', MODEL_PATH, MODEL_PATH.exists())

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print(f'Detected GPUs: {gpu_count}, Tensor Parallel Size: {tp_size}')

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapters),
    max_loras=8 if adapters else 0,
    max_lora_rank=128 if adapters else 0,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key='EMPTY',
)
print('Model registry created successfully.')


In [ ]:
# ==============================================================================
# CELL 3: RUN CANARY EVALUATION ON 3 TARGET BENCHMARK TASKS
# ==============================================================================
import asyncio
import concurrent.futures
import json
import os
import shutil
from pathlib import Path
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

# Target 3 canary tasks: requests_7205 (previously corrupted), fastapi_15588, rich_3521
TARGET_IDS = ['requests_7205', 'fastapi_15588', 'rich_3521']
canary_tasks = [t for t in tasks if t.instance_id in TARGET_IDS]
if len(canary_tasks) < 3:
    print(f'Warning: only found {len(canary_tasks)} matching target IDs; falling back to first 3 tasks.')
    canary_tasks = tasks[:3]

print(f'Canary Tasks to Evaluate ({len(canary_tasks)}): {[t.instance_id for t in canary_tasks]}')

limits, gen_constraints = build_submission_limits()
RESULTS_DIR = WORKING_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = RESULTS_DIR / 'task_results.jsonl'

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=60,
    max_time_minutes=4.0,
    max_tool_calls=30,
    max_turns=50,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2,
        token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=True,
)
evaluator = Evaluator(eval_config)

resolved_count = 0
total_evaluated = 0

print('\nStarting canary evaluation (Testing thinking_level=minimal & Pseudo-XML prompt)...\n', flush=True)

for idx, task in enumerate(canary_tasks, start=1):
    print(f'[{idx}/{len(canary_tasks)}] Evaluating {task.instance_id} ({task.repo})...', flush=True)
    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(canary_tasks))
        is_resolved = bool(result.resolved)
        if is_resolved:
            resolved_count += 1
        total_evaluated += 1

        record = {
            'task_id': task.instance_id,
            'repo': task.repo,
            'resolved': is_resolved,
            'test_exit_code': result.test_exit_code,
            'tool_calls': result.tool_calls,
            'duration_seconds': round(result.duration_seconds, 2),
            'patch_length': len(result.agent_patch or ''),
            'patch': result.agent_patch or '',
        }

        with open(JSONL_PATH, 'a', encoding='utf-8') as jf:
            jf.write(json.dumps(record) + '\n')

        print(
            f'  -> resolved={is_resolved} (Running Score: {resolved_count}/{total_evaluated}), '
            f'exit={result.test_exit_code}, patch={record["patch_length"]}ch, calls={result.tool_calls}, {result.duration_seconds:.1f}s',
            flush=True
        )
    except Exception as e:
        print(f'  -> ERROR evaluating {task.instance_id}: {e}', flush=True)
    finally:
        for sb_dir in Path('/tmp').glob('swegemma_sandbox_*'):
            try:
                shutil.rmtree(sb_dir, ignore_errors=True)
            except Exception:
                pass

print(f'\n=================== CANARY RUN COMPLETE ===================', flush=True)
print(f'Canary Score: {resolved_count}/{total_evaluated} resolved.', flush=True)


In [ ]:
# ==============================================================================
# CELL 4: TRACE INSPECTION & 500-ERROR AUDIT
# ==============================================================================
import json
from pathlib import Path

TRACES_DIR = RESULTS_DIR / 'traces'
print('Examining generated trace files:')
for tf in sorted(TRACES_DIR.glob('*.json')):
    data = json.loads(tf.read_text(encoding='utf-8'))
    steps = data.get('steps', [])
    print(f'\nTrace {tf.name}: {len(steps)} steps')
    for s in steps:
        tc = s.get('tool_calls', [])
        if tc:
            for c in tc:
                fn = c.get('function_name') or c.get('function', {}).get('name')
                print(f'  - Tool Call: {fn}')
        msg = s.get('message', '')
        if '<|"|>' in str(msg) or '<|"|>' in str(tc):
            print('  ⚠️ ALERT: Found delimiter corruption <|"|> in trace!')

print('\nCanary test execution and audit complete.')
